# Notebook 03 — Robustness, Bootstrap & Regression Diagnostics
**Study:** ALV in AI-Mediated L2 Academic Writing — Moderating Role of AI Agency on ODP.

**Purpose.** A full assumption audit and robustness program for the moderation model *ODP = b0 + b1·ALV + b2·AI_Agency + b3·(ALV × AI_Agency) + e*:
1. Multicollinearity (VIF / tolerance; centered predictors);
2. Homoscedasticity (Breusch–Pagan & White tests);
3. Normality of residuals (Shapiro–Wilk & Jarque–Bera);
4. Independence (Durbin–Watson);
5. HC3 robust standard errors vs. classical OLS SEs;
6. Non-parametric bootstrap (5,000 resamples; percentile & BCa 95% CIs) for main effects and the interaction;
7. Sensitivity/influence (Cook's D > 4/N, DFBETAs);
8. Post-hoc power (Cohen's f², α = .05, N = 50).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv('/mnt/data/cleaned_dataset_alv.csv')
n = len(df); print(f'Analytic N = {n}')

## 1. Model specification & centered moderation term
Predictors are mean-centered prior to product-term construction to reduce non-essential collinearity (Aiken & West, 1991).

In [ ]:
df['ALV_c'] = df['ALV'] - df['ALV'].mean()
df['AIA_c'] = df['AI_Agency'] - df['AI_Agency'].mean()
df['interaction'] = df['ALV_c'] * df['AIA_c']

import statsmodels.api as sm
X = sm.add_constant(df[['ALV_c', 'AIA_c', 'interaction']])
y = df['ODP']
model = sm.OLS(y, X).fit()
print(model.summary())

## 2. Multicollinearity: VIF & tolerance
VIF > 5 (tolerance < .20) indicates potentially harmful collinearity; centered main effects plus a product term typically yield acceptable VIFs.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif = pd.DataFrame({
    'Variable': X.columns[1:],
    'VIF': [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]})
vif['Tolerance'] = 1 / vif['VIF']
vif['Flag (>5)'] = np.where(vif['VIF'] > 5, 'FLAG', 'ok')
print(vif.round(3).to_string(index=False))

## 3. Homoscedasticity: Breusch–Pagan & White tests
H₀: constant error variance. p < .05 suggests heteroscedasticity, motivating the HC3 correction in Section 4.

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
bp = het_breuschpagan(model.resid, X)
wh = het_white(model.resid, X)
print(pd.DataFrame({'Test': ['Breusch-Pagan', 'White'],
                    'LM statistic': [bp[0], wh[0]],
                    'p-value': [bp[1], wh[1]],
                    'Conclusion': ['heteroscedastic' if p < .05 else 'homoscedastic'
                                   for p in [bp[1], wh[1]]]}).round(4).to_string(index=False))

## 4. Normality of residuals & independence
Shapiro–Wilk is preferred for N < 50 per case; Jarque–Bera supplements with skew/kurtosis-based evidence. Durbin–Watson ≈ 2 indicates uncorrelated errors (cross-sectional data).

In [ ]:
from statsmodels.stats.stattools import durbin_watson, jarque_bera
sw = stats.shapiro(model.resid)
jb = jarque_bera(model.resid)
dw = durbin_watson(model.resid)
print(f'Shapiro-Wilk:  W = {sw.statistic:.4f}, p = {sw.pvalue:.4f}')
print(f'Jarque-Bera:   JB = {jb[0]:.4f}, p = {jb[1]:.4f}')
print(f'Durbin-Watson: {dw:.3f}  (2.0 = no autocorrelation)')

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sm.qqplot(model.resid, line='45', ax=ax[0]); ax[0].set_title('Q-Q plot of residuals')
ax[1].scatter(model.fittedvalues, model.resid, alpha=.7)
ax[1].axhline(0, ls='--', c='r'); ax[1].set_xlabel('Fitted'); ax[1].set_ylabel('Residual')
ax[1].set_title('Residuals vs. fitted')
fig.tight_layout(); fig.savefig('/mnt/data/fig03_diagnostics.png', dpi=150); plt.show()

## 5. Robust (HC3) standard errors vs. classical OLS
HC3 (Long & Ervin, 2000) is recommended for N ≤ 250. Inference is compared; material SE inflation signals heteroscedasticity-sensitive conclusions.

In [ ]:
rob = model.get_robustcov_results(cov_type='HC3')
comp = pd.DataFrame({
    'OLS b': model.params, 'OLS SE': model.bse, 'OLS p': model.pvalues,
    'HC3 SE': rob.bse, 'HC3 p': rob.pvalues})
comp['SE change (%)'] = 100 * (comp['HC3 SE'] - comp['OLS SE']) / comp['OLS SE']
print(comp.round(4).to_string())

## 6. Non-parametric bootstrap (5,000 resamples)
Case-resampling bootstrap with percentile and BCa 95% CIs for b1 (ALV), b2 (AI_Agency), and b3 (ALV × AI_Agency interaction). BCa adjusts for bias and acceleration; percentile CIs are reported alongside for transparency.

In [ ]:
rng = np.random.default_rng(2024)
B = 5000
data = df[['ALV_c', 'AIA_c', 'interaction', 'ODP']].values
boot = np.empty((B, 3))
for b in range(B):
    idx = rng.integers(0, n, n)
    Xb = sm.add_constant(data[idx, :3])
    fitb = sm.OLS(data[idx, 3], Xb).fit()
    boot[b] = fitb.params[1:4]

from scipy.stats import bootstrap as sci_boot  # available for BCa on precomputed stats
names = ['b1 (ALV)', 'b2 (AI_Agency)', 'b3 (ALV x AIA)']
rows = []
for j, nm in enumerate(names):
    est = model.params[1:][j]
    lo_p, hi_p = np.percentile(boot[:, j], [2.5, 97.5])
    # BCa via jackknife acceleration
    theta_hat = est
    jack = np.empty(n)
    for i in range(n):
        idx = np.delete(np.arange(n), i)
        Xj = sm.add_constant(data[idx, :3])
        jack[i] = sm.OLS(data[idx, 3], Xj).fit().params[1:][j]
    jack_mean = jack.mean()
    a = ((jack_mean - jack)**3).sum() / (6 * (((jack_mean - jack)**2).sum())**1.5)
    p0 = (boot[:, j] < theta_hat).mean()
    z0 = stats.norm.ppf(p0)
    def z_of(x): return stats.norm.cdf(x)
    al1, al2 = z_of(z0 + (z0 - 1.96)/(1 - a*(z0 - 1.96))), z_of(z0 + (z0 + 1.96)/(1 - a*(z0 + 1.96)))
    lo_b, hi_b = np.percentile(boot[:, j], [100*al1, 100*al2])
    rows.append([nm, round(est, 3), (round(lo_p, 3), round(hi_p, 3)),
                 (round(lo_b, 3), round(hi_b, 3)),
                 'sig' if lo_b * hi_b > 0 else 'n.s.'])
print(pd.DataFrame(rows, columns=['Term', 'OLS estimate',
      '95% percentile CI', '95% BCa CI', 'BCa decision']).to_string(index=False))

## 7. Sensitivity & influence: Cook's D and DFBETAs
Cook's D > 4/N and |DFBETA| > 2/√N flag cases whose deletion materially shifts estimates.

In [ ]:
infl = model.get_influence()
cooks_d = infl.cooks_distance[0]
dfb = infl.dfbetas
cutD, cutB = 4/n, 2/np.sqrt(n)
print(f"Cook's D > {cutD:.3f}: {(cooks_d > cutD).sum} case(s) flagged".replace(').sum case', ') case') if False else
      f"Cook's D > {cutD:.3f}: {(cooks_d > cutD).sum()} case(s)")
print(f'|DFBETA| > {cutB:.3f} on any predictor: '
      f'{(np.abs(dfb) > cutB).any(axis=1).sum()} case(s)')
flagged = np.where((cooks_d > cutD) | (np.abs(dfb) > cutB).any(axis=1))[0]
print('Flagged participant_ids:', df['participant_id'].iloc[flagged].tolist())

fig, ax = plt.subplots(figsize=(6, 4))
ax.stem(np.arange(n), cooks_d, markerfmt=',')
ax.axhline(cutD, color='r', ls='--', label=f'4/N = {cutD:.3f}')
ax.set_xlabel('Case index'); ax.set_ylabel("Cook's D"); ax.legend()
fig.tight_layout(); fig.savefig('/mnt/data/fig03_cooks_d.png', dpi=150); plt.show()

## 8. Post-hoc power and sensitivity analysis
Power is computed for the overall model via the noncentral *F* distribution using Cohen's f² = R²/(1 − R²), α = .05, N = 50. A sensitivity analysis reports the minimum detectable effect size (f²) at power = .80.

In [ ]:
R2 = model.rsquared
f2 = R2 / (1 - R2)
u, v = int(model.df_model), int(model.df_resid)
lam = f2 * (u + v + 1)
fcrit = stats.f.ppf(0.95, u, v)
power = 1 - stats.ncf.cdf(fcrit, u, v, lam)
print(f"R2 = {R2:.3f}  ->  Cohen's f2 = {f2:.3f}")
print(f'Post-hoc observed power (alpha = .05, N = 50) = {power:.3f}')

# Sensitivity: minimum detectable f2 at power = .80
from scipy.optimize import brentq
def pw(f2s):
    l = f2s * (u + v + 1)
    return 1 - stats.ncf.cdf(fcrit, u, v, l) - 0.80
f2_min = brentq(pw, 1e-4, 3)
print(f'Minimum detectable f2 at power = .80: {f2_min:.3f} '
      f'(small = .02, medium = .15, large = .35)')

## 9. Summary for the manuscript (APA 7th style)
The cells above produce every quantity required for the robustness subsection: VIFs, BP/White tests, SW/JB statistics, DW statistic, OLS-vs-HC3 comparison, bootstrap CIs, influence flags, and power. Copy the printed tables directly into blind-review-ready text.